<a href="https://colab.research.google.com/github/Afnankh00/Prediction-of-Product-Sales/blob/main/project1p5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# load data and import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load data
fpath = "/content/sales_predictions_2023.csv"
df = pd.read_csv(fpath)

In [3]:
# Remove duplicates
df = df.drop_duplicates()

# Standardize categorical variables
df['Item_Fat_Content'] = df['Item_Fat_Content'].replace({
    'low fat': 'Low Fat',
    'LF': 'Low Fat',
    'reg': 'Regular'
})

In [4]:
# Drop high cardinality features and target
X = df.drop(['Item_Identifier', 'Item_Outlet_Sales'], axis=1)
y = df['Item_Outlet_Sales']

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
# Numerical pipeline with imputation and scaling
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical pipeline with imputation and encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [7]:
# Create full pipeline with model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Fit the model
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Item_Weight',
                                                   'Item_Visibility',
                                                   'Item_MRP',
                                                   'Outlet_Establishment_Year']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Item_Fat_Content',
                                                   'Item_Type',
                                                   'Outlet_Identifier',
                                                   'Outlet_Size',
                                                   'Outlet_Location_Type',
                                                   'Outlet_Type'])])),
                ('regressor', LinearRegression())])

In [8]:
# Make predictions
y_pred = model.predict(X_test)

# Evaluate
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

RMSE: 1069.37
R² Score: 0.5793


In [9]:
# Custom evaluation function for both train and test
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """Evaluate model on both train and test sets"""

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    # Print results
    print(f"\n{'='*60}")
    print(f"{model_name}")
    print(f"{'='*60}")
    print(f"TRAIN - R²: {train_r2:.4f} | RMSE: {train_rmse:.2f} | MAE: {train_mae:.2f}")
    print(f"TEST  - R²: {test_r2:.4f} | RMSE: {test_rmse:.2f} | MAE: {test_mae:.2f}")
    print(f"{'='*60}")

    # Overfitting check
    gap = train_r2 - test_r2
    if gap > 0.1:
        print(f"OVERFITTING detected (Gap: {gap:.4f})")
    elif train_r2 < 0.5 and test_r2 < 0.5:
        print(f"UNDERFITTING detected (Both scores low)")
    else:
        print(f"Good balance (Gap: {gap:.4f})")

    return {
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'gap': gap
    }

##  Evaluate Linear Regression (Train and Test)

In [10]:
# Evaluate Linear Regression on both train and test
lr_metrics = evaluate_model(
    model,
    X_train, y_train,
    X_test, y_test,
    "LINEAR REGRESSION MODEL"
)

# Store for comparison
results = {'Linear Regression': lr_metrics}


LINEAR REGRESSION MODEL
TRAIN - R²: 0.5595 | RMSE: 1141.53 | MAE: 847.22
TEST  - R²: 0.5793 | RMSE: 1069.37 | MAE: 792.02
Good balance (Gap: -0.0198)


# Build Random Forest Model (Default)

In [11]:
from sklearn.ensemble import RandomForestRegressor

# Create Random Forest pipeline
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
        n_estimators=100
    ))
])

# Fit the model
rf_model.fit(X_train, y_train)

# Evaluate
rf_metrics = evaluate_model(
    rf_model,
    X_train, y_train,
    X_test, y_test,
    "RANDOM FOREST MODEL (DEFAULT)"
)

# Store for comparison
results['Random Forest (Default)'] = rf_metrics


RANDOM FOREST MODEL (DEFAULT)
TRAIN - R²: 0.9377 | RMSE: 429.37 | MAE: 298.02
TEST  - R²: 0.5673 | RMSE: 1084.52 | MAE: 757.84
OVERFITTING detected (Gap: 0.3704)


# Compare Linear Regression vs Random Forest

In [12]:
# Model Comparison
print("\n" + "="*60)
print("MODEL COMPARISON (TEST SET)")
print("="*60)
print(f"{'Model':<30} {'R²':<10} {'RMSE':<10}")
print("-"*60)
print(f"{'Linear Regression':<30} {lr_metrics['test_r2']:<10.4f} {lr_metrics['test_rmse']:<10.2f}")
print(f"{'Random Forest (Default)':<30} {rf_metrics['test_r2']:<10.4f} {rf_metrics['test_rmse']:<10.2f}")
print("="*60)

if rf_metrics['test_r2'] > lr_metrics['test_r2']:
    print(" The best: Random Forest (Higher R²)")
else:
    print(" The best: Linear Regression (Higher R²)")


MODEL COMPARISON (TEST SET)
Model                          R²         RMSE      
------------------------------------------------------------
Linear Regression              0.5793     1069.37   
Random Forest (Default)        0.5673     1084.52   
 The best: Linear Regression (Higher R²)


# GridSearchCV for Hyperparameter Tuning

In [13]:
from sklearn.model_selection import GridSearchCV

# Define parameter grid (tuning 2+ hyperparameters)
param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [10, 20, None],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2]
}

# Create GridSearchCV
grid_search = GridSearchCV(
    estimator=Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
    ]),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

# Fit on training data
print("Starting GridSearchCV... ")
grid_search.fit(X_train, y_train)

# Best parameters
print("\n" + "="*60)
print("GRIDSEARCHCV RESULTS")
print("="*60)
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV R² Score: {grid_search.best_score_:.4f}")

Starting GridSearchCV... 
Fitting 5 folds for each of 36 candidates, totalling 180 fits

GRIDSEARCHCV RESULTS
Best Parameters: {'regressor__max_depth': 10, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 100}
Best CV R² Score: 0.5853


# Evaluate Tuned Random Forest Model

In [14]:
# Get the best model
best_rf_model = grid_search.best_estimator_

# Evaluate tuned model
tuned_metrics = evaluate_model(
    best_rf_model,
    X_train, y_train,
    X_test, y_test,
    "RANDOM FOREST MODEL (TUNED)"
)

# Store for comparison
results['Random Forest (Tuned)'] = tuned_metrics

# Compare with default RF
print("\n" + "="*60)
print("TUNING IMPACT")
print("="*60)
print(f"Default RF Test R²:  {rf_metrics['test_r2']:.4f}")
print(f"Tuned RF Test R²:    {tuned_metrics['test_r2']:.4f}")
print(f"Improvement:         {tuned_metrics['test_r2'] - rf_metrics['test_r2']:.4f}")
print("="*60)


RANDOM FOREST MODEL (TUNED)
TRAIN - R²: 0.7163 | RMSE: 916.08 | MAE: 648.51
TEST  - R²: 0.6037 | RMSE: 1037.83 | MAE: 726.16
OVERFITTING detected (Gap: 0.1126)

TUNING IMPACT
Default RF Test R²:  0.5673
Tuned RF Test R²:    0.6037
Improvement:         0.0365


#Final Model Comparison

In [15]:
# Create comparison DataFrame

comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (Default)', 'Random Forest (Tuned)'],
    'Train R²': [lr_metrics['train_r2'], rf_metrics['train_r2'], tuned_metrics['train_r2']],
    'Test R²': [lr_metrics['test_r2'], rf_metrics['test_r2'], tuned_metrics['test_r2']],
    'Train RMSE': [lr_metrics['train_rmse'], rf_metrics['train_rmse'], tuned_metrics['train_rmse']],
    'Test RMSE': [lr_metrics['test_rmse'], rf_metrics['test_rmse'], tuned_metrics['test_rmse']],
    'Train MAE': [lr_metrics['train_mae'], rf_metrics['train_mae'], tuned_metrics['train_mae']],
    'Test MAE': [lr_metrics['test_mae'], rf_metrics['test_mae'], tuned_metrics['test_mae']],
    'Gap (R²)': [lr_metrics['gap'], rf_metrics['gap'], tuned_metrics['gap']]
})

# Display
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
display(comparison_df.round(4))


FINAL MODEL COMPARISON


,Model,Train R²,Test R²,Train RMSE,Test RMSE,Train MAE,Test MAE,Gap (R²)
0,Linear Regression,0.5595,0.5793,1141.5316,1069.3653,847.2209,792.0250,-0.0198
1,Random Forest (Default),0.9377,0.5673,429.3684,1084.5238,298.0231,757.8359,0.3704
2,Random Forest (Tuned),0.7163,0.6037,916.0772,1037.8311,648.5134,726.1589,0.1126
